In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
player_play = pd.read_csv("data/player_play.csv")
game_tracking_2023 = pd.read_csv("data/game_tracking_2023.csv")
combine_results = pd.read_csv("data/combine_results.csv")

# Gather our plays

First we will filter based on if the play included an offensive lineman

Second we will filter out for only passing plays, as we are looking at pass protection

In [3]:
passing = player_play.loc[
    (player_play['lined_up_position'] == 'T') |
    (player_play['lined_up_position'] == 'G') |
    (player_play['lined_up_position'] == 'C'),
    ['game_id', 'play_id', 'nfl_id', 'lined_up_position', 'pass_result']
]

passing = passing[passing['pass_result'].notna()]

In [4]:
passing

,game_id,play_id,nfl_id,lined_up_position,pass_result
0,2025111610,2279,57183,T,I
8,2023091801,56,55976,T,IN
10,2025122701,63,57183,T,C
16,2023101503,3590,55973,G,C
21,2026010405,2568,57238,C,C
...,...,...,...,...,...
314167,2025092200,986,57330,G,R
314172,2025110300,3012,58214,G,I
314182,2025122811,2539,57207,G,C
314183,2025081052,373,58211,T,I


# Gather the tracking

For this example we will again be examining Darnell Wright and Paris Johnson Jr.

In [15]:
wright_plays = game_tracking_2023[game_tracking_2023['nfl_id'] == 55874]
paris_plays = game_tracking_2023[game_tracking_2023 ['nfl_id'] == 55870]

In [6]:
wright_weight = combine_results.loc[
    combine_results['nfl_id'] == 55874,
    'combine_weight'
].iloc[0]

paris_weight = combine_results.loc[
    combine_results['nfl_id'] == 55870,
    'combine_weight'
].iloc[0]

In [7]:
wright_plays['force'] = (
    wright_plays['a'] * (wright_weight / 2.20462)
).round(2)

paris_plays['force'] = (
    paris_plays['a'] * (paris_weight / 2.20462)
).round(2)

In [18]:
wright_plays = wright_plays.sort_values(
    ['game_id', 'play_id', 'time']
).copy()

In [20]:
wright_plays['row_num'] = (
    wright_plays.groupby(['game_id', 'play_id']).cumcount()
)

# identify when snap occurs
snap_rows = (
    wright_plays[wright_plays['event'] == 'ball_snap']
    [['game_id', 'play_id', 'row_num']]
    .rename(columns={'row_num': 'snap_row'})
)

wright_plays = wright_plays.merge(
    snap_rows,
    on=['game_id', 'play_id'],
    how='left'
)


In [22]:
wright_snap_3sec = wright_plays[
    wright_plays['row_num'].between(
        wright_plays['snap_row'],
        wright_plays['snap_row'] + 30
    )
].copy()

# Passing

Our focus here is on passing, and in order to find these passing plays we must identify every play where there is a pass

In [31]:
wright_passing = wright_snap_3sec.merge(
    passing[passing['nfl_id'] == 55874],
    on = ['game_id', 'play_id'],
    how = 'left'
)

In [35]:
wright_passing

,game_id,play_id,nfl_id_x,time,x,y,s,a,dis,o,dir,event,row_num,snap_row,nfl_id_y,lined_up_position,pass_result,seconds_after_snap
0,2023081251,477,55874,2023-08-12T17:22:04.100,85.86,26.93,0.23,1.07,0.02,263.66,143.01,ball_snap,10,10.0,55874.0,T,C,0.0
1,2023081251,477,55874,2023-08-12T17:22:04.200,85.88,26.91,0.36,1.13,0.03,262.80,144.64,NaN,11,10.0,55874.0,T,C,0.1
2,2023081251,477,55874,2023-08-12T17:22:04.300,85.92,26.86,0.58,1.44,0.06,262.80,144.62,NaN,12,10.0,55874.0,T,C,0.2
3,2023081251,477,55874,2023-08-12T17:22:04.400,85.96,26.80,0.77,1.26,0.08,262.08,144.62,NaN,13,10.0,55874.0,T,C,0.3
4,2023081251,477,55874,2023-08-12T17:22:04.500,86.01,26.72,0.90,1.04,0.09,260.44,146.76,NaN,14,10.0,55874.0,T,C,0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34989,2024010706,2920,55874,2024-01-07T23:43:46.500,55.80,17.73,1.78,0.97,0.18,7.06,146.60,NaN,35,9.0,55874.0,T,I,2.6
34990,2024010706,2920,55874,2024-01-07T23:43:46.600,55.91,17.59,1.77,0.98,0.18,18.37,141.46,NaN,36,9.0,55874.0,T,I,2.7
34991,2024010706,2920,55874,2024-01-07T23:43:46.700,56.03,17.45,1.75,0.95,0.18,26.21,138.58,NaN,37,9.0,55874.0,T,I,2.8
34992,2024010706,2920,55874,2024-01-07T23:43:46.800,56.15,17.34,1.68,1.11,0.17,34.53,133.48,pass_outcome_incomplete,38,9.0,55874.0,T,I,2.9


In [34]:
wright_passing['seconds_after_snap'] = (
    wright_passing['row_num'] - wright_passing['snap_row']
) * 0.1

In [38]:
time_groups = [1.0, 1.5, 2.0, 2.5, 3.0]

average_force = []

for seconds in time_groups:
    group = wright_passing[
        wright_passing['seconds_after_snap'] <= seconds
    ]

    average_force.append({
        'Time Interval': f'0.0 - {seconds}',
        'Average Acceleration': group['a'].mean()
    })

average_force_df = pd.DataFrame(average_force)

In [40]:
average_force_df['Force'] = average_force_df['Average Acceleration'] * (wright_weight / 2.20462)

`average_force_df` is for Darnell Wright only

In [41]:
average_force_df

,Time Interval,Average Acceleration,Force
0,0.0 - 1.0,1.411429,213.191398
1,0.0 - 1.5,1.353276,204.407563
2,0.0 - 2.0,1.372269,207.276359
3,0.0 - 2.5,1.404458,212.138423
4,0.0 - 3.0,1.433350,216.502457


### Future updates

1. Need to filter out screen passes as blocking is not a factor here
2. Need to factor out qb_kneel plays as well
3. Shorten plays so that if there is a pass or a sack you stop counting the values